In [ ]:
import random, math, csv
import numpy as np
from typing import List
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import plotly.graph_objects as go
from torch.nn.utils import spectral_norm

SEED        = 43
EPOCHS      = 30
BATCH_SIZE  = 128

LR           = 1e-4
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-3

ALPHA_CE     = 1.0
HINGE_WEIGHT = 4.0
RECON_WEIGHT = 3.0

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0
PERSISTENT  = False
PIN_MEMORY  = (DEVICE == "cuda")

MARGIN        = 1.0
SIGMA_Q       = 0.4
ALPHA         = 0.2
CLS_SCALE     = 1.0

SIGMA_MIN_GRID = 0.0
SIGMA_MAX_GRID = 2.0
N_SIGMA_GRID   = 21
SIGMA_GRID     = np.linspace(SIGMA_MIN_GRID, SIGMA_MAX_GRID, N_SIGMA_GRID)

NUM_CLASSES = 10
LATENT_DIM  = 64

SMOOTH_WINDOW_EPOCHS = 3

def set_seed(seed: int = 43):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

class AELinearMNIST(nn.Module):
    def __init__(self, latent_dim: int = LATENT_DIM, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.flatten = nn.Flatten()

        self.encoder = nn.Sequential(
            nn.Linear(1 * 28 * 28, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, latent_dim),
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 1 * 28 * 28),
        )

        self.classifier = spectral_norm(nn.Linear(latent_dim, num_classes))

    def encode(self, x):
        x_flat = self.flatten(x)
        z = self.encoder(x_flat)
        return z

    def decode(self, z):
        x_flat_rec = self.decoder(z)
        x_rec = x_flat_rec.view(-1, 1, 28, 28)
        return x_rec

    def forward(self, x):
        z = self.encode(x)
        logits = self.classifier(z)
        x_rec = self.decode(z)
        return CLS_SCALE * logits, x_rec

    @torch.no_grad()
    def logits_from_x_nograd(self, x):
        z = self.encode(x)
        logits = self.classifier(z)
        return CLS_SCALE * logits

    @torch.no_grad()
    def features_from_x_nograd(self, x):
        return self.encode(x)

def cross_entropy_from_logits(logits, y, label_smoothing: float = 0.1):
    return F.cross_entropy(logits, y, reduction='mean', label_smoothing=label_smoothing)

def margin_hinge_loss_from_logits(logits, targets, margin=1.0):
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float('inf')
    max_other = masked.max(dim=1).values
    m = logits[idx, targets] - max_other
    v = F.relu(margin - m)
    return v.mean()

@torch.no_grad()
def samplewise_margin_and_hinge(logits, targets, margin=1.0):
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float('inf')
    max_other = masked.max(dim=1).values
    m = logits[idx, targets] - max_other
    h = F.relu(margin - m)
    return m, h

MNIST_MEAN = (0.1307,)
MNIST_STD  = (0.3081,)

transform_train = transforms.Compose([
    transforms.RandomCrop(28, padding=4),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

transform_test  = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MNIST_MEAN, MNIST_STD),
])

train_set = datasets.MNIST(root='./data', train=True,  download=True, transform=transform_train)
test_set  = datasets.MNIST(root='./data', train=False, download=True, transform=transform_test)

def make_half_splits(dataset, batch_size=128):
    n = len(dataset)
    idx_cal = np.arange(0, n//2)
    idx_tst = np.arange(n//2, n)
    ds_cal  = Subset(dataset, idx_cal)
    ds_tst  = Subset(dataset, idx_tst)
    ld_cal  = DataLoader(ds_cal, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    ld_tst  = DataLoader(ds_tst, batch_size=batch_size, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    return ld_cal, ld_tst

P_cal_loader, P_tst_loader = make_half_splits(test_set, BATCH_SIZE)
Q_cal_loader, Q_tst_loader = P_cal_loader, P_tst_loader

def apply_Q_transform(x: torch.Tensor, sigma: float) -> torch.Tensor:
    if sigma <= 0.0:
        return x

    B = x.size(0)
    device = x.device

    contrast_scale = 1.0 + 0.8 * sigma * (2.0 * torch.rand(B, 1, 1, 1, device=device) - 1.0)
    brightness_shift = 0.4 * sigma * (2.0 * torch.rand(B, 1, 1, 1, device=device) - 1.0)
    x = x * contrast_scale + brightness_shift

    max_shift = int(4 * sigma)
    if max_shift > 0:
        shift_x = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        shift_y = torch.randint(-max_shift, max_shift + 1, (B,), device=device)
        for i in range(B):
            x[i] = torch.roll(x[i], shifts=(int(shift_y[i]), int(shift_x[i])), dims=(1, 2))

    noise_std = 0.1 * sigma
    if noise_std > 0:
        x = x + noise_std * torch.randn_like(x)

    return x

model = AELinearMNIST(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

milestone1 = int(EPOCHS * 0.5)
milestone2 = int(EPOCHS * 0.75)
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[milestone1, milestone2],
    gamma=0.1,
)

def train_one_epoch(epoch: int):
    model.train()
    sums = {'cce':0.0,'hinge':0.0,'recon':0.0,'acc':0.0,'n':0}
    pbar = tqdm(DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT),
                desc=f"Train {epoch:02d}")
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, x_rec = model(x)
        cce   = cross_entropy_from_logits(logits, y, label_smoothing=0.1)
        hinge = margin_hinge_loss_from_logits(logits, y, margin=MARGIN)
        recon = F.mse_loss(x_rec, x)
        loss = ALPHA_CE * cce + HINGE_WEIGHT * hinge + RECON_WEIGHT * recon

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            acc = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        sums['cce']   += cce.item()*bs
        sums['hinge'] += hinge.item()*bs
        sums['recon'] += recon.item()*bs
        sums['acc']   += acc*bs
        sums['n']     += bs
        pbar.set_postfix({
            'cce':   f"{sums['cce']/sums['n']:.4f}",
            'hinge': f"{sums['hinge']/sums['n']:.4f}",
            'recon': f"{sums['recon']/sums['n']:.4f}",
            'acc':   f"{sums['acc']/sums['n']:.4f}",
        })
    for k in ['cce','hinge','recon','acc']:
        sums[k] /= sums['n']
    return sums

@torch.no_grad()
def evaluate(loader, tag="Eval"):
    model.eval()
    tot = {'cce':0.0,'hinge':0.0,'acc':0.0,'n':0}
    pbar = tqdm(loader, desc=f"{tag}", leave=False)
    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        cce   = F.cross_entropy(logits, y, reduction='mean').item()
        hinge = margin_hinge_loss_from_logits(logits, y, MARGIN).item()
        acc   = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        tot['cce']  += cce*bs
        tot['hinge']+= hinge*bs
        tot['acc']  += acc*bs
        tot['n']    += bs
    for k in ['cce','hinge','acc']:
        tot[k] /= tot['n']
    return tot

@torch.no_grad()
def predictive_entropy_from_logits(logits: torch.Tensor) -> torch.Tensor:
    probs = F.softmax(logits, dim=1)
    log_probs = (probs + 1e-12).log()
    ent = -(probs * log_probs).sum(dim=1)
    return ent

@torch.no_grad()
def scores_for_labels_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> np.ndarray:
    B = logits.size(0)
    idx = torch.arange(B, device=logits.device)
    masked = logits.clone()
    masked[idx, labels] = -float('inf')
    max_other = masked.max(dim=1).values
    s = (max_other - logits[idx, labels]).detach().float().cpu().numpy()
    return s

def conformal_quantile(scores: np.ndarray, alpha: float, finite_sample: bool = True) -> float:
    n = scores.size
    if n == 0:
        return float('nan')
    if finite_sample:
        qlevel = math.ceil((n + 1) * (1 - alpha)) / n
    else:
        qlevel = 1 - alpha
    if qlevel > 1:
        return float('inf')
    try:
        return float(np.quantile(scores, qlevel, method='higher'))
    except TypeError:
        return float(np.quantile(scores, qlevel, interpolation='higher'))

def coverage_given_threshold(scores_true: np.ndarray, threshold: float) -> float:
    if scores_true.size == 0:
        return float('nan')
    return float((scores_true <= threshold).mean())

@torch.no_grad()
def collect_true_scores(loader: DataLoader, sigma: float) -> np.ndarray:
    model.eval()
    all_scores = []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        s = scores_for_labels_from_logits(logits, y)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

@torch.no_grad()
def expected_set_size_threshold(loader: DataLoader, sigma: float, q: float) -> float:
    model.eval()
    total_size = 0.0
    total_n = 0

    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        B, K = logits.shape

        values, indices = logits.topk(k=2, dim=1)
        max1 = values[:, 0:1]
        max2 = values[:, 1:2]
        argmax = indices[:, 0]

        classes = torch.arange(K, device=logits.device).unsqueeze(0).expand(B, -1)
        argmax_exp = argmax.unsqueeze(1).expand(-1, K)
        max_other = torch.where(classes == argmax_exp, max2, max1)

        scores_all = max_other - logits
        set_sizes = (scores_all <= q).sum(dim=1).float()

        total_size += set_sizes.sum().item()
        total_n += B

    return total_size / total_n if total_n > 0 else float('nan')

@torch.no_grad()
def collect_pseudo_scores_mixed(
    loader: DataLoader,
    sigma: float,
    u: float,
    num_classes: int = NUM_CLASSES,
) -> np.ndarray:
    model.eval()
    all_scores = []
    for x, _ in loader:
        x = x.to(DEVICE)
        x = apply_Q_transform(x, sigma)
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)
        yhat = logits.argmax(dim=1)

        B, K = logits.shape
        rand_labels = torch.randint(low=0, high=num_classes, size=(B,), device=logits.device)
        mask_low = (ent <= u)
        pseudo_y = torch.where(mask_low, yhat, rand_labels)

        s = scores_for_labels_from_logits(logits, pseudo_y)
        all_scores.append(s)
    return np.concatenate(all_scores, axis=0) if all_scores else np.array([])

@torch.no_grad()
def compute_uP_star(
    P_cal_loader: DataLoader,
    P_tst_loader: DataLoader,
    alpha: float = ALPHA,
    num_u_points: int = 30,
):
    ent_all = []
    for x, _ in P_cal_loader:
        x = x.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        ent = predictive_entropy_from_logits(logits)
        ent_all.append(ent.cpu().numpy())
    ent_all = np.concatenate(ent_all, axis=0)

    low_q, high_q = 0.05, 0.95
    t_min = np.quantile(ent_all, low_q)
    t_max = np.quantile(ent_all, high_q)
    u_cutoffs = np.linspace(t_min, t_max, num_u_points)

    scores_P_tst_true = collect_true_scores(P_tst_loader, sigma=0.0)

    cov_P_vals = []
    target_cov = 1.0 - alpha

    u_P_star = u_cutoffs[0]
    q_Phat_uP_star = None

    for u in u_cutoffs:
        scores_P_cal_pseudo = collect_pseudo_scores_mixed(
            loader=P_cal_loader,
            sigma=0.0,
            u=u,
            num_classes=NUM_CLASSES,
        )
        q_Phat_u = conformal_quantile(scores_P_cal_pseudo, alpha, finite_sample=True)
        cov_P_u = coverage_given_threshold(scores_P_tst_true, q_Phat_u)
        cov_P_vals.append(cov_P_u)

        if cov_P_u >= target_cov:
            u_P_star = u
            q_Phat_uP_star = q_Phat_u

    cov_P_vals = np.array(cov_P_vals)

    if q_Phat_uP_star is None:
        scores_P_cal_pseudo = collect_pseudo_scores_mixed(
            loader=P_cal_loader,
            sigma=0.0,
            u=u_cutoffs[0],
            num_classes=NUM_CLASSES,
        )
        q_Phat_uP_star = conformal_quantile(scores_P_cal_pseudo, alpha, finite_sample=True)
        u_P_star = float(u_cutoffs[0])

    return float(u_P_star), u_cutoffs, cov_P_vals, float(q_Phat_uP_star)

hinge_P_epochs         = np.zeros(EPOCHS, dtype=np.float64)
cov_Q_uP_epochs_sigma  = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)
cov_P_uP_epochs        = np.zeros(EPOCHS, dtype=np.float64)
u_P_star_epochs        = np.zeros(EPOCHS, dtype=np.float64)

ESS_P_uP_epochs        = np.zeros(EPOCHS, dtype=np.float64)
ESS_P_oracle_epochs    = np.zeros(EPOCHS, dtype=np.float64)
ESS_Q_uP_epochs_sigma      = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)
ESS_Q_oracle_epochs_sigma  = np.zeros((EPOCHS, N_SIGMA_GRID), dtype=np.float64)

def smooth_1d(a: np.ndarray, window: int = 3) -> np.ndarray:
    a = np.asarray(a, dtype=np.float64)
    if window <= 1 or a.size == 0 or a.size < window:
        return a.copy()
    pad_left = window // 2
    pad_right = window - 1 - pad_left
    a_pad = np.pad(a, (pad_left, pad_right), mode="edge")
    kernel = np.ones(window, dtype=np.float64) / float(window)
    return np.convolve(a_pad, kernel, mode="valid")

def smooth_2d_along_epochs(Z: np.ndarray, window: int = 3) -> np.ndarray:
    Z = np.asarray(Z, dtype=np.float64)
    if window <= 1 or Z.size == 0 or Z.shape[0] < window:
        return Z.copy()
    Z_sm = np.empty_like(Z)
    for j in range(Z.shape[1]):
        Z_sm[:, j] = smooth_1d(Z[:, j], window)
    return Z_sm

In [ ]:
from baselines import train_vae, ecp_sets, wqlcp_sets, coverage_and_size
import csv
import math
import random
from contextlib import contextmanager

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.lines import Line2D
from torch.utils.data import DataLoader, Subset

try:
    import scienceplots

    plt.style.use(["science", "ieee", "no-latex"])
except ImportError:
    plt.style.use("default")

REPEATS = 5
BASE_REPEAT_SEED = 20260412
TABLE_SIGMAS = [0.0, 0.3, 0.7, 1.0, 1.2, 1.6, 1.8, 2.0]
SAVE_PREFIX = "mnist_multirun_mean_std_fullplots"
N_U_POINTS = 30
RHO_TRIM_QUANTILE = 0.9
EPS = 1e-12

required_globals = [
    "AELinearMNIST",
    "train_set",
    "test_set",
    "DEVICE",
    "NUM_CLASSES",
    "LATENT_DIM",
    "BATCH_SIZE",
    "NUM_WORKERS",
    "PIN_MEMORY",
    "PERSISTENT",
    "EPOCHS",
    "LR",
    "MOMENTUM",
    "WEIGHT_DECAY",
    "ALPHA_CE",
    "HINGE_WEIGHT",
    "RECON_WEIGHT",
    "MARGIN",
    "ALPHA",
    "SIGMA_GRID",
    "SIGMA_Q",
    "apply_Q_transform",
]
missing = [name for name in required_globals if name not in globals()]
if missing:
    raise RuntimeError(
        "This cell expects the base MNIST setup cell to have run first. "
        f"Missing globals: {missing}"
    )

mpl.rcParams["font.size"] = 14
mpl.rcParams["axes.labelsize"] = 14
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["legend.fontsize"] = 9
mpl.rcParams["xtick.labelsize"] = 13
mpl.rcParams["ytick.labelsize"] = 13
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
plt.rcParams["lines.linewidth"] = 1.0

COLOR_SOURCE = "tab:red"
COLOR_HARD = "tab:blue"
COLOR_SOURCE_TUNED = "tab:green"
COLOR_ECP = "tab:brown"
COLOR_WQLCP = "tab:purple"
COLOR_LB = "k"

color_cov_base = "tab:blue"
color_cov_tau = "tab:orange"
color_cov_oracle = "tab:cyan"
color_LB = "k"

IEEE_WIDTH_1COL = 3.5
IEEE_ASPECT = 0.9
FIGSIZE_1COL = (IEEE_WIDTH_1COL, IEEE_WIDTH_1COL * IEEE_ASPECT)

def _set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@contextmanager
def _temporary_seed(seed: int):
    py_state = random.getstate()
    np_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

    _set_all_seeds(seed)
    try:
        yield
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.random.set_rng_state(torch_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)

def _cross_entropy_from_logits(logits, y, label_smoothing: float = 0.1):
    return F.cross_entropy(logits, y, reduction="mean", label_smoothing=label_smoothing)

def _margin_hinge_loss_from_logits(logits, targets, margin=1.0):
    idx = torch.arange(logits.size(0), device=logits.device)
    masked = logits.clone()
    masked[idx, targets] = -float("inf")
    max_other = masked.max(dim=1).values
    margins = logits[idx, targets] - max_other
    return F.relu(margin - margins).mean()

def _make_random_half_splits(dataset, batch_size: int, split_seed: int):
    n = len(dataset)
    rng = np.random.default_rng(split_seed)
    perm = rng.permutation(n)
    half = n // 2
    idx_cal = perm[:half]
    idx_tst = perm[half:]

    ds_cal = Subset(dataset, idx_cal.tolist())
    ds_tst = Subset(dataset, idx_tst.tolist())

    ld_cal = DataLoader(
        ds_cal,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT,
    )
    ld_tst = DataLoader(
        ds_tst,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT,
    )
    return ld_cal, ld_tst

def _build_model_optimizer_scheduler():
    model = AELinearMNIST(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LR,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    milestone1 = int(EPOCHS * 0.5)
    milestone2 = int(EPOCHS * 0.75)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[milestone1, milestone2],
        gamma=0.1,
    )
    return model, optimizer, scheduler

def _make_train_loader(epoch_seed: int):
    generator = torch.Generator()
    generator.manual_seed(epoch_seed)
    return DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT,
    )

def _train_one_epoch(model, optimizer, epoch_seed: int):
    _set_all_seeds(epoch_seed)
    model.train()
    loader = _make_train_loader(epoch_seed)
    sums = {"cce": 0.0, "hinge": 0.0, "recon": 0.0, "acc": 0.0, "n": 0}

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits, x_rec = model(x)
        cce = _cross_entropy_from_logits(logits, y, label_smoothing=0.1)
        hinge = _margin_hinge_loss_from_logits(logits, y, margin=MARGIN)
        recon = F.mse_loss(x_rec, x)
        loss = ALPHA_CE * cce + HINGE_WEIGHT * hinge + RECON_WEIGHT * recon

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            acc = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        sums["cce"] += cce.item() * bs
        sums["hinge"] += hinge.item() * bs
        sums["recon"] += recon.item() * bs
        sums["acc"] += acc * bs
        sums["n"] += bs

    for key in ["cce", "hinge", "recon", "acc"]:
        sums[key] /= max(sums["n"], 1)
    return sums

@torch.no_grad()
def _evaluate_clean(model, loader):
    model.eval()
    tot = {"cce": 0.0, "hinge": 0.0, "acc": 0.0, "n": 0}

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model.logits_from_x_nograd(x)
        cce = F.cross_entropy(logits, y, reduction="mean").item()
        hinge = _margin_hinge_loss_from_logits(logits, y, MARGIN).item()
        acc = (logits.argmax(1) == y).float().mean().item()
        bs = x.size(0)
        tot["cce"] += cce * bs
        tot["hinge"] += hinge * bs
        tot["acc"] += acc * bs
        tot["n"] += bs

    for key in ["cce", "hinge", "acc"]:
        tot[key] /= max(tot["n"], 1)
    return tot

def _entropy_from_logits_np(logits: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    probs /= probs.sum(axis=1, keepdims=True)
    return -(probs * np.log(probs + 1e-12)).sum(axis=1)

def _scores_for_labels_from_logits_np(logits: np.ndarray, labels: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    rows = np.arange(logits.shape[0])
    masked = logits.copy()
    masked[rows, labels] = -np.inf
    max_other = masked.max(axis=1)
    return max_other - logits[rows, labels]

def _samplewise_margins_np(logits: np.ndarray, labels: np.ndarray) -> np.ndarray:
    logits = np.asarray(logits, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    rows = np.arange(logits.shape[0])
    masked = logits.copy()
    masked[rows, labels] = -np.inf
    max_other = masked.max(axis=1)
    return logits[rows, labels] - max_other

def _hinge_loss_from_margins_np(margins: np.ndarray) -> float:
    if margins.size == 0:
        return 0.0
    return float(np.mean(np.maximum(1.0 - margins, 0.0)))

def _ramp_loss_from_margins_np(margins: np.ndarray) -> float:
    if margins.size == 0:
        return 0.0
    hinge = np.maximum(1.0 - margins, 0.0)
    ramp = np.minimum(hinge, 1.0)
    return float(np.mean(ramp))

def _conformal_quantile_np(scores: np.ndarray, alpha: float) -> float:
    scores = np.asarray(scores, dtype=np.float64)
    n = scores.size
    if n == 0:
        return float("nan")
    qlevel = math.ceil((n + 1) * (1.0 - float(alpha))) / n
    if qlevel > 1.0:
        return float("inf")
    try:
        return float(np.quantile(scores, qlevel, method="higher"))
    except TypeError:
        return float(np.quantile(scores, qlevel, interpolation="higher"))

def _coverage_np(scores_true: np.ndarray, q: float) -> float:
    return float(np.mean(np.asarray(scores_true) <= float(q)))

def _expected_set_size_from_logits_np(logits: np.ndarray, q: float) -> float:
    logits = np.asarray(logits, dtype=np.float64)
    top_sorted = np.sort(logits, axis=1)
    top1 = top_sorted[:, -1][:, None]
    top2 = top_sorted[:, -2][:, None]
    argmax = np.argmax(logits, axis=1)
    classes = np.arange(logits.shape[1])[None, :]
    max_other = np.where(classes == argmax[:, None], top2, top1)
    scores_all = max_other - logits
    return float(np.mean((scores_all <= float(q)).sum(axis=1)))

@torch.no_grad()
def _collect_eval_arrays(model, loader, sigma: float, seed: int):
    model.eval()
    mean_t = torch.tensor(MNIST_MEAN, device=DEVICE).view(1, -1, 1, 1)
    std_t = torch.tensor(MNIST_STD, device=DEVICE).view(1, -1, 1, 1)
    all_logits = []
    all_labels = []
    all_recon = []

    with _temporary_seed(seed):
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            if float(sigma) != 0.0:
                x = apply_Q_transform(x, float(sigma))

            logits = model.logits_from_x_nograd(x)
            raw = x * std_t + mean_t
            rec, _, _ = vae(raw, sample=False)
            recon = (rec - raw).square().flatten(1).sum(1)
            all_logits.append(logits.detach().cpu().numpy().astype(np.float32))
            all_labels.append(y.detach().cpu().numpy().astype(np.int64))
            all_recon.append(recon.detach().cpu().numpy().astype(np.float64))

    return (
        np.concatenate(all_logits, axis=0),
        np.concatenate(all_labels, axis=0),
        np.concatenate(all_recon, axis=0),
    )

@torch.no_grad()
def _estimate_feature_rho_mix(model, loader, sigma: float, seed: int, trim_quantile: float = 0.9):
    model.eval()
    all_dists = []

    with _temporary_seed(seed):
        for x, _ in loader:
            x = x.to(DEVICE)
            z = model.features_from_x_nograd(x)
            x_shift = apply_Q_transform(x.clone(), float(sigma))
            z_shift = model.features_from_x_nograd(x_shift)
            d = (z - z_shift).norm(p=2, dim=1)
            all_dists.append(d.detach().cpu().numpy())

    if not all_dists:
        return float("nan"), float("nan")

    dists = np.concatenate(all_dists, axis=0)
    rho_full = float(dists.mean())

    if trim_quantile is None:
        rho_trim = rho_full
    else:
        q = np.quantile(dists, trim_quantile)
        mask = dists <= q
        rho_trim = float(dists[mask].mean()) if np.any(mask) else rho_full

    return rho_full, rho_trim

def _estimate_L_gamma_feature_global(model) -> float:
    with torch.no_grad():
        W = model.classifier.weight.detach()
        diff = W.unsqueeze(0) - W.unsqueeze(1)
        norms = diff.norm(p=2, dim=2)
        Kc = norms.size(0)
        mask = ~torch.eye(Kc, dtype=torch.bool, device=norms.device)
        L_gamma = norms[mask].max().item()
    return float(L_gamma)

def _sample_random_labels(n: int, num_classes: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.integers(low=0, high=num_classes, size=n, dtype=np.int64)

def _compute_u_star(
    p_cal_logits: np.ndarray,
    p_cal_entropy: np.ndarray,
    p_tst_true_scores: np.ndarray,
    rand_labels_p: np.ndarray,
    alpha: float,
    num_u_points: int,
):
    t_min = float(np.quantile(p_cal_entropy, 0.05))
    t_max = float(np.quantile(p_cal_entropy, 0.95))
    if np.isclose(t_min, t_max):
        u_grid = np.array([t_min], dtype=np.float64)
    else:
        u_grid = np.linspace(t_min, t_max, num_u_points)

    hard_labels = np.argmax(p_cal_logits, axis=1)
    target_cov = 1.0 - float(alpha)
    best_u = float(u_grid[0])
    best_q = None

    for u in u_grid:
        pseudo_labels = np.where(p_cal_entropy <= u, hard_labels, rand_labels_p)
        pseudo_scores = _scores_for_labels_from_logits_np(p_cal_logits, pseudo_labels)
        q_u = _conformal_quantile_np(pseudo_scores, alpha)
        cov_u = _coverage_np(p_tst_true_scores, q_u)
        if cov_u >= target_cov:
            best_u = float(u)
            best_q = float(q_u)

    if best_q is None:
        best_u = float(u_grid[0])
        pseudo_labels = np.where(p_cal_entropy <= best_u, hard_labels, rand_labels_p)
        pseudo_scores = _scores_for_labels_from_logits_np(p_cal_logits, pseudo_labels)
        best_q = _conformal_quantile_np(pseudo_scores, alpha)

    return best_u, float(best_q)

def _summarize(metric_matrix: np.ndarray):
    mean = metric_matrix.mean(axis=0)
    if metric_matrix.shape[0] > 1:
        std = metric_matrix.std(axis=0, ddof=1)
    else:
        std = np.zeros(metric_matrix.shape[1], dtype=np.float64)
    return mean, std

nominal_cov = 1.0 - ALPHA
sigma_vals = SIGMA_GRID.astype(float)

coverage_plot_runs = {
    "source": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "hard": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "tuned": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "ecp": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "wqlcp": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
}
bound_runs = {
    "lb_ramp_q": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "lb_ramp_prho": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
}
coverage_table_runs = {
    "source": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "hard": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "tuned": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "ecp": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "wqlcp": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "oracle": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
}
ess_table_runs = {
    "source": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "hard": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "tuned": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "ecp": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "wqlcp": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "oracle": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
}
coverage_tau_runs = {
    "hard": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "tau": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "oracle": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
}
ess_tau_runs = {
    "hard": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "tau": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
    "oracle": np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64),
}
lb_tau_runs = np.zeros((REPEATS, len(SIGMA_GRID)), dtype=np.float64)
u_star_runs = np.zeros(REPEATS, dtype=np.float64)

print("=" * 80)
print(f"Running repeated MNIST experiment for mean/std with {REPEATS} repeats")
print("Each repeat uses a fresh random calibration/test split and a fresh retraining run.")
print("=" * 80)

vae = train_vae(datasets.MNIST(root="./data", train=True, download=True, transform=transforms.ToTensor()), DEVICE, seed=20261112)

for rep in range(REPEATS):
    repeat_seed = BASE_REPEAT_SEED + 1000 * rep
    split_seed = repeat_seed + 1

    print(f"\n=== Repeat {rep + 1}/{REPEATS} | seed={repeat_seed} | split_seed={split_seed} ===")

    _set_all_seeds(repeat_seed)
    P_cal_loader, P_tst_loader = _make_random_half_splits(test_set, BATCH_SIZE, split_seed)
    Q_cal_loader, Q_tst_loader = P_cal_loader, P_tst_loader

    model, optimizer, scheduler = _build_model_optimizer_scheduler()

    for epoch in range(1, EPOCHS + 1):
        epoch_seed = repeat_seed + 100 + epoch
        tr = _train_one_epoch(model, optimizer, epoch_seed=epoch_seed)
        teP = _evaluate_clean(model, P_tst_loader)
        scheduler.step()

        if epoch <= 5 or epoch == EPOCHS or epoch % 5 == 0:
            print(
                f"Repeat {rep + 1:02d} | Epoch {epoch:03d} | "
                f"Train CE {tr['cce']:.4f} hinge {tr['hinge']:.4f} recon {tr['recon']:.4f} acc {tr['acc']:.4f} | "
                f"Test(P) CE {teP['cce']:.4f} hinge {teP['hinge']:.4f} acc {teP['acc']:.4f}"
            )

    P_cal_logits, P_cal_labels, P_cal_recon = _collect_eval_arrays(
        model, P_cal_loader, sigma=0.0, seed=repeat_seed + 10000
    )
    P_tst_logits, P_tst_labels, P_tst_recon = _collect_eval_arrays(
        model, P_tst_loader, sigma=0.0, seed=repeat_seed + 10001
    )

    P_cal_true_scores = _scores_for_labels_from_logits_np(P_cal_logits, P_cal_labels)
    P_tst_true_scores = _scores_for_labels_from_logits_np(P_tst_logits, P_tst_labels)
    P_cal_entropy = _entropy_from_logits_np(P_cal_logits)
    P_tst_margins = _samplewise_margins_np(P_tst_logits, P_tst_labels)

    q_source = _conformal_quantile_np(P_cal_true_scores, ALPHA)
    rand_labels_p = _sample_random_labels(len(P_cal_logits), NUM_CLASSES, seed=repeat_seed + 10002)
    u_star, _ = _compute_u_star(
        p_cal_logits=P_cal_logits,
        p_cal_entropy=P_cal_entropy,
        p_tst_true_scores=P_tst_true_scores,
        rand_labels_p=rand_labels_p,
        alpha=float(ALPHA),
        num_u_points=N_U_POINTS,
    )
    u_star_runs[rep] = u_star

    P_cal_hard_labels = np.argmax(P_cal_logits, axis=1)
    scores_P_cal_pseudo = _scores_for_labels_from_logits_np(P_cal_logits, P_cal_hard_labels)
    q_pseudo_P = _conformal_quantile_np(scores_P_cal_pseudo, ALPHA)
    coverage_P_pseudo = _coverage_np(P_tst_true_scores, q_pseudo_P)
    Delta_P = float(nominal_cov - coverage_P_pseudo)
    L_h_P = _hinge_loss_from_margins_np(P_tst_margins)
    denom_base = float(L_h_P - Delta_P)

    L_r_source = _ramp_loss_from_margins_np(P_tst_margins)
    L_gamma_global = _estimate_L_gamma_feature_global(model)

    print(
        f"Repeat {rep + 1}: u_P^*={u_star:.6f}, q_source={q_source:.6f}, "
        f"Delta_P={Delta_P:.6f}, L_h(P)={L_h_P:.6f}, L_r(P)={L_r_source:.6f}, L_gamma={L_gamma_global:.6f}"
    )

    for s_idx, sigma in enumerate(SIGMA_GRID):
        sigma_val = float(sigma)

        Q_cal_logits, Q_cal_labels, Q_cal_recon = _collect_eval_arrays(
            model, Q_cal_loader, sigma=sigma_val, seed=repeat_seed + 20000 + s_idx
        )
        Q_tst_logits, Q_tst_labels, Q_tst_recon = _collect_eval_arrays(
            model, Q_tst_loader, sigma=sigma_val, seed=repeat_seed + 30000 + s_idx
        )

        Q_cal_true_scores = _scores_for_labels_from_logits_np(Q_cal_logits, Q_cal_labels)
        Q_tst_true_scores = _scores_for_labels_from_logits_np(Q_tst_logits, Q_tst_labels)
        Q_cal_entropy = _entropy_from_logits_np(Q_cal_logits)
        Q_tst_entropy = _entropy_from_logits_np(Q_tst_logits)

        Q_cal_hard_labels = np.argmax(Q_cal_logits, axis=1)
        Q_cal_hard_scores = _scores_for_labels_from_logits_np(Q_cal_logits, Q_cal_hard_labels)

        q_hard = _conformal_quantile_np(Q_cal_hard_scores, ALPHA)
        q_oracle = _conformal_quantile_np(Q_cal_true_scores, ALPHA)

        cov_source = _coverage_np(Q_tst_true_scores, q_source)
        ess_source = _expected_set_size_from_logits_np(Q_tst_logits, q_source)

        cov_hard = _coverage_np(Q_tst_true_scores, q_hard)
        ess_hard = _expected_set_size_from_logits_np(Q_tst_logits, q_hard)

        rand_labels_q = _sample_random_labels(len(Q_cal_logits), NUM_CLASSES, seed=repeat_seed + 40000 + s_idx)
        q_tuned_labels = np.where(Q_cal_entropy <= u_star, Q_cal_hard_labels, rand_labels_q)
        q_tuned_scores = _scores_for_labels_from_logits_np(Q_cal_logits, q_tuned_labels)
        q_tuned = _conformal_quantile_np(q_tuned_scores, ALPHA)
        cov_tuned = _coverage_np(Q_tst_true_scores, q_tuned)
        ess_tuned = _expected_set_size_from_logits_np(Q_tst_logits, q_tuned)

        cov_ecp, ess_ecp = coverage_and_size(ecp_sets(P_cal_logits, P_cal_labels, Q_tst_logits, ALPHA), Q_tst_labels)
        cov_wqlcp, ess_wqlcp = coverage_and_size(
            wqlcp_sets(P_cal_logits, P_cal_labels, P_cal_recon, Q_tst_logits, Q_tst_recon, ALPHA), Q_tst_labels
        )

        Q_tst_margins = _samplewise_margins_np(Q_tst_logits, Q_tst_labels)
        L_h_Q = _hinge_loss_from_margins_np(Q_tst_margins)
        if abs(denom_base) <= EPS:
            tau_raw = 0.0
        else:
            tau_raw = 2.0 * (L_h_Q / denom_base - 1.0)
        tau = float(max(0.0, tau_raw))
        denom = 1.0 + tau / 2.0
        lb_tau = float(nominal_cov - (L_h_Q / denom)) if denom > 0 else float("nan")
        q_hard_tau = q_hard + tau
        cov_tau = _coverage_np(Q_tst_true_scores, q_hard_tau)
        ess_tau = _expected_set_size_from_logits_np(Q_tst_logits, q_hard_tau)

        cov_oracle = _coverage_np(Q_tst_true_scores, q_oracle)
        ess_oracle = _expected_set_size_from_logits_np(Q_tst_logits, q_oracle)

        rho_full, rho_trim = _estimate_feature_rho_mix(
            model,
            P_cal_loader,
            sigma=sigma_val,
            seed=repeat_seed + 50000 + s_idx,
            trim_quantile=RHO_TRIM_QUANTILE,
        )
        L_r_Q = _ramp_loss_from_margins_np(Q_tst_margins)
        cov_lb_ramp_Q = np.clip(nominal_cov - L_r_Q, 0.0, 1.0)
        cov_lb_ramp_Prho = np.clip(nominal_cov - (L_r_source + L_gamma_global * rho_trim), 0.0, 1.0)

        coverage_plot_runs["source"][rep, s_idx] = cov_source
        coverage_plot_runs["hard"][rep, s_idx] = cov_hard
        coverage_plot_runs["tuned"][rep, s_idx] = cov_tuned
        coverage_plot_runs["ecp"][rep, s_idx] = cov_ecp
        coverage_plot_runs["wqlcp"][rep, s_idx] = cov_wqlcp

        bound_runs["lb_ramp_q"][rep, s_idx] = cov_lb_ramp_Q
        bound_runs["lb_ramp_prho"][rep, s_idx] = cov_lb_ramp_Prho

        coverage_table_runs["source"][rep, s_idx] = cov_source
        coverage_table_runs["hard"][rep, s_idx] = cov_hard
        coverage_table_runs["tuned"][rep, s_idx] = cov_tuned
        coverage_table_runs["ecp"][rep, s_idx] = cov_ecp
        coverage_table_runs["wqlcp"][rep, s_idx] = cov_wqlcp
        coverage_table_runs["oracle"][rep, s_idx] = cov_oracle

        ess_table_runs["source"][rep, s_idx] = ess_source
        ess_table_runs["hard"][rep, s_idx] = ess_hard
        ess_table_runs["tuned"][rep, s_idx] = ess_tuned
        ess_table_runs["ecp"][rep, s_idx] = ess_ecp
        ess_table_runs["wqlcp"][rep, s_idx] = ess_wqlcp
        ess_table_runs["oracle"][rep, s_idx] = ess_oracle

        coverage_tau_runs["hard"][rep, s_idx] = cov_hard
        coverage_tau_runs["tau"][rep, s_idx] = cov_tau
        coverage_tau_runs["oracle"][rep, s_idx] = cov_oracle

        ess_tau_runs["hard"][rep, s_idx] = ess_hard
        ess_tau_runs["tau"][rep, s_idx] = ess_tau
        ess_tau_runs["oracle"][rep, s_idx] = ess_oracle
        lb_tau_runs[rep, s_idx] = lb_tau

    sig_print_idx = int(np.argmin(np.abs(SIGMA_GRID - float(SIGMA_Q))))
    print(
        f"Repeat {rep + 1} summary at sigma={float(SIGMA_GRID[sig_print_idx]):.2f}: "
        f"source={coverage_plot_runs['source'][rep, sig_print_idx]:.4f}, "
        f"hard={coverage_plot_runs['hard'][rep, sig_print_idx]:.4f}, "
        f"tuned={coverage_plot_runs['tuned'][rep, sig_print_idx]:.4f}, "
        f"ecp={coverage_plot_runs['ecp'][rep, sig_print_idx]:.4f}, "
        f"wqlcp={coverage_plot_runs['wqlcp'][rep, sig_print_idx]:.4f}, "
        f"oracle={coverage_table_runs['oracle'][rep, sig_print_idx]:.4f}"
    )

summary_rows = []
for s_idx, sigma in enumerate(SIGMA_GRID):
    row = {
        "sigma": float(sigma),
        "nominal_coverage": float(nominal_cov),
    }

    for key in ["source", "hard", "tuned", "ecp", "wqlcp"]:
        mean_vals, std_vals = _summarize(coverage_plot_runs[key][:, [s_idx]])
        row[f"{key}_coverage_mean"] = float(mean_vals[0])
        row[f"{key}_coverage_std"] = float(std_vals[0])

    for key in ["source", "hard", "tuned", "ecp", "wqlcp", "oracle"]:
        cov_mean, cov_std = _summarize(coverage_table_runs[key][:, [s_idx]])
        ess_mean, ess_std = _summarize(ess_table_runs[key][:, [s_idx]])
        row[f"{key}_coverage_table_mean"] = float(cov_mean[0])
        row[f"{key}_coverage_table_std"] = float(cov_std[0])
        row[f"{key}_ess_mean"] = float(ess_mean[0])
        row[f"{key}_ess_std"] = float(ess_std[0])

    for key in ["hard", "tau", "oracle"]:
        cov_mean, cov_std = _summarize(coverage_tau_runs[key][:, [s_idx]])
        ess_mean, ess_std = _summarize(ess_tau_runs[key][:, [s_idx]])
        row[f"{key}_tauplot_coverage_mean"] = float(cov_mean[0])
        row[f"{key}_tauplot_coverage_std"] = float(cov_std[0])
        row[f"{key}_tauplot_ess_mean"] = float(ess_mean[0])
        row[f"{key}_tauplot_ess_std"] = float(ess_std[0])

    lb_tau_mean, lb_tau_std = _summarize(lb_tau_runs[:, [s_idx]])
    lb_rq_mean, lb_rq_std = _summarize(bound_runs["lb_ramp_q"][:, [s_idx]])
    lb_rprho_mean, lb_rprho_std = _summarize(bound_runs["lb_ramp_prho"][:, [s_idx]])
    row["lb_tau_mean"] = float(lb_tau_mean[0])
    row["lb_tau_std"] = float(lb_tau_std[0])
    row["lb_ramp_q_mean"] = float(lb_rq_mean[0])
    row["lb_ramp_q_std"] = float(lb_rq_std[0])
    row["lb_ramp_prho_mean"] = float(lb_rprho_mean[0])
    row["lb_ramp_prho_std"] = float(lb_rprho_std[0])

    summary_rows.append(row)

csv_path = f"{SAVE_PREFIX}_summary.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

def _fmt_pm(mean_val: float, std_val: float, scale: float = 1.0) -> str:
    return f"${scale * mean_val:.2f} \\\\pm {scale * std_val:.2f}$"

table_rows = []
for sigma_req in TABLE_SIGMAS:
    s_idx = int(np.argmin(np.abs(SIGMA_GRID - sigma_req)))
    sigma_used = float(SIGMA_GRID[s_idx])

    cov_src_mean, cov_src_std = _summarize(coverage_table_runs["source"][:, [s_idx]])
    ess_src_mean, ess_src_std = _summarize(ess_table_runs["source"][:, [s_idx]])
    cov_hard_mean, cov_hard_std = _summarize(coverage_table_runs["hard"][:, [s_idx]])
    ess_hard_mean, ess_hard_std = _summarize(ess_table_runs["hard"][:, [s_idx]])
    cov_tuned_mean, cov_tuned_std = _summarize(coverage_table_runs["tuned"][:, [s_idx]])
    ess_tuned_mean, ess_tuned_std = _summarize(ess_table_runs["tuned"][:, [s_idx]])
    cov_ecp_mean, cov_ecp_std = _summarize(coverage_table_runs["ecp"][:, [s_idx]])
    ess_ecp_mean, ess_ecp_std = _summarize(ess_table_runs["ecp"][:, [s_idx]])
    cov_wqlcp_mean, cov_wqlcp_std = _summarize(coverage_table_runs["wqlcp"][:, [s_idx]])
    ess_wqlcp_mean, ess_wqlcp_std = _summarize(ess_table_runs["wqlcp"][:, [s_idx]])
    cov_or_mean, cov_or_std = _summarize(coverage_table_runs["oracle"][:, [s_idx]])
    ess_or_mean, ess_or_std = _summarize(ess_table_runs["oracle"][:, [s_idx]])

    table_rows.append(
        {
            "sigma": sigma_used,
            "cov_src": _fmt_pm(cov_src_mean[0], cov_src_std[0], scale=100.0),
            "ess_src": _fmt_pm(ess_src_mean[0], ess_src_std[0]),
            "cov_hard": _fmt_pm(cov_hard_mean[0], cov_hard_std[0], scale=100.0),
            "ess_hard": _fmt_pm(ess_hard_mean[0], ess_hard_std[0]),
            "cov_tuned": _fmt_pm(cov_tuned_mean[0], cov_tuned_std[0], scale=100.0),
            "ess_tuned": _fmt_pm(ess_tuned_mean[0], ess_tuned_std[0]),
            "cov_ecp": _fmt_pm(cov_ecp_mean[0], cov_ecp_std[0], scale=100.0),
            "ess_ecp": _fmt_pm(ess_ecp_mean[0], ess_ecp_std[0]),
            "cov_wqlcp": _fmt_pm(cov_wqlcp_mean[0], cov_wqlcp_std[0], scale=100.0),
            "ess_wqlcp": _fmt_pm(ess_wqlcp_mean[0], ess_wqlcp_std[0]),
            "cov_tgt": _fmt_pm(cov_or_mean[0], cov_or_std[0], scale=100.0),
            "ess_tgt": _fmt_pm(ess_or_mean[0], ess_or_std[0]),
        }
    )

header = r"""\begin{table*}[t]
\centering
\caption{MNIST: coverage and expected set size versus shift level $\sigma$, averaged over %d independent runs with fresh random calibration/test splits and retraining (nominal $1-\alpha=%.2f$). Entries report mean $\pm$ standard deviation for source calibration, hard pseudo-calibration, source-tuned pseudo-calibration, ECP, WQLCP, and target calibration.}
\label{tab:mnist_cov_ess_multirun}
\setlength{\tabcolsep}{3pt}
\small
\begin{tabular}{c|cc|cc|cc|cc|cc|cc}
\hline
 & \multicolumn{2}{c|}{Source cal} & \multicolumn{2}{c|}{Hard pseudo cal} & \multicolumn{2}{c|}{Source-tuned cal} & \multicolumn{2}{c|}{ECP} & \multicolumn{2}{c|}{WQLCP} & \multicolumn{2}{c}{Target cal (ideal)} \\
$\sigma$ & Cov (\%%) & ESS & Cov (\%%) & ESS & Cov (\%%) & ESS & Cov (\%%) & ESS & Cov (\%%) & ESS & Cov (\%%) & ESS \\
\hline
""" % (REPEATS, nominal_cov)

body_lines = []
for row in table_rows:
    body_lines.append(
        f"{row['sigma']:.2f} & "
        f"{row['cov_src']} & {row['ess_src']} & "
        f"{row['cov_hard']} & {row['ess_hard']} & "
        f"{row['cov_tuned']} & {row['ess_tuned']} & "
        f"{row['cov_ecp']} & {row['ess_ecp']} & "
        f"{row['cov_wqlcp']} & {row['ess_wqlcp']} & "
        f"{row['cov_tgt']} & {row['ess_tgt']} \\\\"
    )

footer = r"""
\hline
\end{tabular}
\end{table*}
"""

table_latex = header + "\n".join(body_lines) + footer
print("\n" + "=" * 80)
print("LaTeX table with mean ± std")
print("=" * 80)
print(table_latex)

tex_path = f"{SAVE_PREFIX}_table.tex"
with open(tex_path, "w") as f:
    f.write(table_latex)

fig_cov, ax_cov = plt.subplots(figsize=FIGSIZE_1COL)

for key, color, label in [
    ("source", COLOR_SOURCE, "Source Cal."),
    ("hard", COLOR_HARD, "Hard Pseudo-Cal."),
    ("tuned", COLOR_SOURCE_TUNED, "Source-tuned Pseudo-Cal."),
    ("ecp", COLOR_ECP, "ECP"),
    ("wqlcp", COLOR_WQLCP, "WQLCP"),
]:
    mean_vals, std_vals = _summarize(coverage_plot_runs[key])
    ax_cov.plot(SIGMA_GRID, mean_vals, color=color, linestyle="-", label=label)
    ax_cov.fill_between(
        SIGMA_GRID,
        np.clip(mean_vals - std_vals, 0.0, 1.0),
        np.clip(mean_vals + std_vals, 0.0, 1.0),
        color=color,
        alpha=0.10,
    )

lb_ramp_q_mean, _ = _summarize(bound_runs["lb_ramp_q"])
lb_ramp_prho_mean, _ = _summarize(bound_runs["lb_ramp_prho"])
ax_cov.plot(SIGMA_GRID, lb_ramp_q_mean, color=COLOR_LB, linestyle="--", label=r"$1-\alpha - L_r(f,Q_\sigma)$")
ax_cov.plot(
    SIGMA_GRID,
    lb_ramp_prho_mean,
    color=COLOR_LB,
    linestyle=":",
    label=r"$1-\alpha - (L_r(f,P)+L_\gamma\rho_{\mathrm{mix}}(\sigma))$",
)
ax_cov.axhline(nominal_cov, linestyle="--", color="gray", label=r"Nominal $1-\alpha$")

ax_cov.set_xlabel(r"Shift level $\sigma$")
ax_cov.set_ylabel(r"Coverage on $Q_\sigma$")
ax_cov.set_ylim(0.0, 1.03)
ax_cov.set_xlim(float(SIGMA_GRID.min()), 1.5)
ax_cov.grid(True, which="both", linewidth=0.5, alpha=0.5)
ax_cov.legend(loc='lower left', bbox_to_anchor=(0.14, 0.0), fontsize=7.5, framealpha=0.9, facecolor='white', edgecolor='none', borderaxespad=0.3, handlelength=1.6, labelspacing=0.25)

plt.tight_layout()
cov_bounds_pdf = f"{SAVE_PREFIX}_coverage_vs_sigma_bounds_ecp_wqlcp.pdf"
cov_bounds_png = f"{SAVE_PREFIX}_coverage_vs_sigma_bounds_ecp_wqlcp.png"
plt.savefig(cov_bounds_pdf)
plt.savefig(cov_bounds_png, dpi=300)
plt.show()

fig_tau, ax1 = plt.subplots(figsize=FIGSIZE_1COL)
ax2 = ax1.twinx()

for key, color in [
    ("hard", color_cov_base),
    ("tau", color_cov_tau),
    ("oracle", color_cov_oracle),
]:
    cov_mean, cov_std = _summarize(coverage_tau_runs[key])
    ess_mean, ess_std = _summarize(ess_tau_runs[key])

    ax1.plot(SIGMA_GRID, cov_mean, "-", color=color)
    ax1.fill_between(
        SIGMA_GRID,
        np.clip(cov_mean - cov_std, 0.0, 1.0),
        np.clip(cov_mean + cov_std, 0.0, 1.0),
        color=color,
        alpha=0.10,
    )

    ax2.plot(SIGMA_GRID, ess_mean, ":", color=color)
    ax2.fill_between(
        SIGMA_GRID,
        np.maximum(ess_mean - ess_std, 0.0),
        ess_mean + ess_std,
        color=color,
        alpha=0.08,
    )

lb_tau_mean, _ = _summarize(lb_tau_runs)
ax1.plot(SIGMA_GRID, lb_tau_mean, "--", color=color_LB)

ax1.set_xlabel(r"Shift level $\sigma$")
ax1.set_ylabel(r"Coverage on $Q_\sigma$")
ax1.set_ylim(0.0, 1.02)
ax1.set_xlim(float(SIGMA_GRID.min()), 1.5)
ax1.grid(True, which="both", linewidth=0.5, alpha=0.5)

ax2.set_ylabel(r"Expected set size on $Q_\sigma$")
ax2.set_ylim(0.0, NUM_CLASSES + 0.5)

method_handles = [
    Line2D([0], [0], marker="o", linestyle="None", markersize=6,
           markerfacecolor=color_cov_base, markeredgecolor=color_cov_base),
    Line2D([0], [0], marker="o", linestyle="None", markersize=6,
           markerfacecolor=color_cov_tau, markeredgecolor=color_cov_tau),
    Line2D([0], [0], marker="o", linestyle="None", markersize=6,
           markerfacecolor=color_cov_oracle, markeredgecolor=color_cov_oracle),
]
method_labels = [
    r"$q_{\tilde{Q}_{\sigma,\alpha}}$",
    r"$q_{\tilde{Q}_{\sigma,\alpha}}+\tau(\sigma)$",
    r"$q_{Q_{\sigma,\alpha}}$",
]
ax1.legend(
    method_handles,
    method_labels,
    frameon=False,
    loc="lower left",
    bbox_to_anchor=(0.01, 0.1),
    handletextpad=0.6,
    borderaxespad=0.0,
)

plt.tight_layout()
tau_pdf = f"{SAVE_PREFIX}_coverage_ess_vs_sigma_pseudocal_tau_plus_oracle.pdf"
tau_png = f"{SAVE_PREFIX}_coverage_ess_vs_sigma_pseudocal_tau_plus_oracle.png"
plt.savefig(tau_pdf)
plt.savefig(tau_png, dpi=300)
plt.show()

print(f"\nSaved summary CSV to {csv_path}")
print(f"Saved LaTeX table to {tex_path}")
print(f"Saved coverage+bounds plot to {cov_bounds_pdf} and {cov_bounds_png}")
print(f"Saved tau/oracle coverage+ESS plot to {tau_pdf} and {tau_png}")
print(f"u_P^* across runs: mean={u_star_runs.mean():.4f}, std={u_star_runs.std(ddof=1) if REPEATS > 1 else 0.0:.4f}")